In [15]:
import sys
import json
import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
import random
from typing import List
from sem_proj.data.datasets import BoasDataset


# PROJECT_ROOT = Path(__file__).resolve().parents[1]
PROJECT_ROOT = Path.cwd().parent
TARGET_DIR = PROJECT_ROOT / "plots"

In [2]:
SPLITS_FILE = PROJECT_ROOT / "data" / "processed" / "data_splits_70_15_15.json"
assert SPLITS_FILE.exists(), f"Splits file not found: {SPLITS_FILE}"
with open(SPLITS_FILE, 'r') as f:
    splits = json.load(f)

In [3]:
train_subjects = splits['train_subjects']
val_subjects = splits['val_subjects']

In [4]:
def sample_subjects(subjects: List[str], fraction: float, seed: int = 42) -> List[str]:
    if fraction < 0.0 or fraction > 1.0:
        raise ValueError(f"fraction must be between 0.0 and 1.0, got {fraction}")
    if fraction == 1.0:
        return sorted(subjects)
    rng = random.Random(seed)
    n_sample = max(1, int(len(subjects) * fraction))
    return sorted(rng.sample(subjects, n_sample))

In [5]:
seed = 42 # SEED VALUE HAS TO BE CONSISTENT WITH THE CORRESPONDING EXPERIMENT(s)
proportions = {
    1: [],
    5: [],
    10: [],
    20: [],
    50: [],
    100: []
}
for fraction in np.array(list(proportions.keys()))/100:
    sampled_train = sample_subjects(train_subjects, fraction, seed)
    train_ds = BoasDataset(
        subjects=sampled_train,
        mode='headband',
        preprocess_config=None,
        use_cache=True,
        transform_hb=None 
    )
    labels = []
    for idx in range(0, len(train_ds)):
        ep = train_ds[idx]
        label = ep[1].item()
        labels.append(label)
    c0, c1, c2, c3, c4 = 0, 0, 0, 0, 0
    for idx in range(0, len(labels)):
        c0 += 1 if labels[idx] == 0 else 0
        c1 += 1 if labels[idx] == 1 else 0
        c2 += 1 if labels[idx] == 2 else 0
        c3 += 1 if labels[idx] == 3 else 0
        c4 += 1 if labels[idx] == 4 else 0
    proportion_key = int(fraction * 100)
    proportions[proportion_key] = [round(c0/len(labels), ndigits=4), round(c1/len(labels), ndigits=4), round(c2/len(labels), ndigits=4), round(c3/len(labels), ndigits=4), round(c4/len(labels), ndigits=4)]


Loading sub-89...
  ✓ Loaded sub-89 (headband) from cache
Loading sub-104...
  ✓ Loaded sub-104 (headband) from cache
Loading sub-114...
  ✓ Loaded sub-114 (headband) from cache
Loading sub-3...
  ✓ Loaded sub-3 (headband) from cache
Loading sub-89...
  ✓ Loaded sub-89 (headband) from cache
Loading sub-104...
  ✓ Loaded sub-104 (headband) from cache
Loading sub-113...
  ✓ Loaded sub-113 (headband) from cache
Loading sub-114...
  ✓ Loaded sub-114 (headband) from cache
Loading sub-123...
  ✓ Loaded sub-123 (headband) from cache
Loading sub-22...
  ✓ Loaded sub-22 (headband) from cache
Loading sub-26...
  ✓ Loaded sub-26 (headband) from cache
Loading sub-3...
  ✓ Loaded sub-3 (headband) from cache
Loading sub-89...
  ✓ Loaded sub-89 (headband) from cache
Loading sub-104...
  ✓ Loaded sub-104 (headband) from cache
Loading sub-105...
  ✓ Loaded sub-105 (headband) from cache
Loading sub-111...
  ✓ Loaded sub-111 (headband) from cache
Loading sub-113...
  ✓ Loaded sub-113 (headband) from cach

In [6]:
for key in proportions.keys():
    print(f"Fraction {key}%: Class Proportions {proportions[key]}")

Fraction 1%: Class Proportions [0.1555, 0.036, 0.6066, 0.0103, 0.1916]
Fraction 5%: Class Proportions [0.1119, 0.0335, 0.6406, 0.0445, 0.1695]
Fraction 10%: Class Proportions [0.1386, 0.0351, 0.6167, 0.0496, 0.16]
Fraction 20%: Class Proportions [0.1319, 0.0365, 0.621, 0.0387, 0.1719]
Fraction 50%: Class Proportions [0.1617, 0.0363, 0.6017, 0.0421, 0.1582]
Fraction 100%: Class Proportions [0.1535, 0.037, 0.6069, 0.0393, 0.1633]


In [13]:
fractions = np.array([1, 5, 10, 20, 50, 100])  # in percent
x = np.arange(len(fractions))

prop1 = proportions[1]
prop5 = proportions[5]
prop10 = proportions[10]
prop20 = proportions[20]
prop50 = proportions[50]
prop100 = proportions[100]

P = np.array([prop1, prop5, prop10, prop20, prop50, prop100])
print(P)
print(P.shape)

class_names = ["Wake", "N1", "N2", "N3", "REM"]


[[0.1555 0.036  0.6066 0.0103 0.1916]
 [0.1119 0.0335 0.6406 0.0445 0.1695]
 [0.1386 0.0351 0.6167 0.0496 0.16  ]
 [0.1319 0.0365 0.621  0.0387 0.1719]
 [0.1617 0.0363 0.6017 0.0421 0.1582]
 [0.1535 0.037  0.6069 0.0393 0.1633]]
(6, 5)


In [24]:
# fig, ax = plt.subplots(figsize=(8, 4.5))
fig, ax = plt.subplots(figsize=(10, 6))

bottom = np.zeros(len(fractions))
for j in range(np.size(P, 1)):
    ax.bar(x, P[:, j], bottom=bottom, label=class_names[j])
    bottom += P[:, j]

ax.set_xticks(x)
ax.set_xticklabels([f"{f}%" for f in fractions], fontsize=16)
ax.tick_params(axis='y', labelsize=16)
ax.set_xlabel("\nFraction of labeled training data\n", fontsize=18)
ax.set_ylabel("\nClass distribution\n", fontsize=18)  # proportions sum to 1 within each bar
ax.set_ylim(0, 1)

ax.legend(ncol=3, frameon=False, loc="upper center", bbox_to_anchor=(0.5, 1.18), fontsize=16)
ax.grid(axis="y", alpha=0.3)

# plt.tight_layout()
plt.savefig(TARGET_DIR / "class_distr_varying_label_fraction.pdf", dpi=300, bbox_inches='tight')